# WikiRate Raw Data Processing

=============================================================
 Fashion Sustainability Dataset — Data Organisation Script
=============================================================
 Source:  WikiRate / Fashion Revolution
          What Fuels Fashion — Fashion Transparency Index 2024
 License: CC BY 4.0

 Input:   Raw CSV exported from WikiRate (long format)
 Output:  Clean Excel workbook with 3 sheets:
            1. Full Dataset       – wide format, colour-coded
            2. Cluster Input      – theme scores for K-means
            3. By Sub-segment     – segment-level summary

 Usage:
   1. Place this script in the same folder as your CSV
   2. Update INPUT_FILE below to match your filename
   3. Run:  python organise_fashion_dataset.py
=============================================================


In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

## FILE PATHS — update these


In [ ]:
INPUT_FILE = 'Wikirate-2026_04_21_103953-Answer.csv'   # your WikiRate CSV
OUTPUT_FILE = 'Fashion_Sustainability_Dataset.xlsx'

# ============================================================
# STEP 1 — HELPER FUNCTIONS
# ============================================================

def fill(hex_color):
    return PatternFill('solid', fgColor=hex_color)

def font(size=10, bold=False, color='000000'):
    return Font(name='Arial', size=size, bold=bold, color=color)

def align(horizontal='center', wrap=False):
    return Alignment(horizontal=horizontal, vertical='center', wrap_text=wrap)

def thin_border():
    s = Side(style='thin', color='BDBDBD')
    return Border(left=s, right=s, top=s, bottom=s)

def header_border():
    thick  = Side(style='medium', color='1A3C5E')
    thin   = Side(style='thin',   color='BDBDBD')
    return Border(left=thin, right=thin, top=thin, bottom=thick)

# Colour palette
COLOURS = {
    'navy':     '1A3C5E',
    'blue':     '2E6DA4',
    'lt_blue':  'D6E4F0',
    'teal':     '00695C',
    'green':    '1B5E20',
    'lt_green': 'C8E6C9',
    'red_lt':   'FFCDD2',
    'yellow':   'FFF9C4',
    'grey':     'F5F5F5',
    'white':    'FFFFFF',
    'slate':    '37474F',
}

# ============================================================
# STEP 2 — LOAD & CLEAN DATA
# ============================================================

print("Loading data...")

# WikiRate CSV has 4 metadata rows at the top — skip them
df = pd.read_csv(INPUT_FILE, skiprows=4)

print(f"  Rows loaded:    {len(df)}")
print(f"  Companies:      {df['Company'].nunique()}")
print(f"  Metrics:        {df['Metric'].nunique()}")

# Remove "Fashion Revolution+" prefix from metric names
df['Metric_clean'] = df['Metric'].str.replace('Fashion Revolution\\+', '', regex=True)

# Pivot from long format → wide format (one row per company)
pivot = df.pivot_table(
    index='Company',
    columns='Metric_clean',
    values='Value',
    aggfunc='first'
).reset_index()

print(f"  Pivoted shape:  {pivot.shape}")

# ============================================================
# STEP 3 — ENCODE YES/NO → 1/0
# ============================================================

def encode_value(val):
    """Convert Yes → 1, No → 0, keep other text values as-is."""
    if pd.isna(val):
        return ''
    v = str(val).strip().lower()
    if v == 'yes': return 1
    if v == 'no':  return 0
    return val  # keep non-binary text values unchanged

for col in pivot.columns:
    if col != 'Company':
        pivot[col] = pivot[col].apply(encode_value)

# ============================================================
# STEP 4 — DEFINE THEMATIC SECTIONS
# ============================================================
# Each section has a hex colour and list of keywords to match
# column names against.

SECTIONS = {
    'Carbon & Emissions': (
        '1B5E20',
        ['Carbon', 'Scope', 'Emission', 'Decarboni',
         'Science Based', 'Fossil Fuel', 'Commitment to Degrowth']
    ),
    'Energy & Renewables': (
        '0D47A1',
        ['Energy', 'Renewable', 'Electricity', 'RE100', 'Electrify']
    ),
    'Supply Chain & Transparency': (
        'BF360C',
        ['Supply', 'Supplier', 'manufacturing', 'processing',
         'raw material', 'Open Supply']
    ),
    'Social & Workers': (
        '880E4F',
        ['Living Wage', 'Worker', 'Collective Bargaining',
         'Trade Union', 'Piece Rate', 'Minimum Wage',
         'democratically elected']
    ),
    'Governance & Strategy': (
        '37474F',
        ['Executive Pay', 'Just Transition', 'Affected Stakeholder',
         'Climate-Related Risk', 'Climate Strategy']
    ),
    'Materials & Products': (
        '4A148C',
        ['Material', 'Fibre', 'Products Produced']
    ),
}

# Build ordered column list and section lookup
used_cols   = {'Company'}
ordered_cols = ['Company']
section_map  = {}  # col_name → (section_name, hex_color)

for sec_name, (sec_color, keywords) in SECTIONS.items():
    for col in pivot.columns:
        if col not in used_cols and any(k in col for k in keywords):
            ordered_cols.append(col)
            section_map[col] = (sec_name, sec_color)
            used_cols.add(col)

# Any remaining columns not matched go into "Other"
for col in pivot.columns:
    if col not in used_cols:
        ordered_cols.append(col)
        section_map[col] = ('Other', COLOURS['slate'])

pivot = pivot[ordered_cols]

# ============================================================
# STEP 5 — COMPUTE SCORES
# ============================================================

# Identify binary (0/1) columns only
binary_cols = [
    c for c in pivot.columns
    if c != 'Company'
    and set(pivot[c].dropna().unique()).issubset({0, 1, ''})
]

# Overall disclosure score = % of binary metrics answered "Yes"
pivot['Overall Disclosure Score (%)'] = pivot[binary_cols].apply(
    lambda row: round(row[row != ''].sum() / len(binary_cols) * 100, 1),
    axis=1
)

# Theme-level scores
for sec_name, (sec_color, keywords) in SECTIONS.items():
    sec_bin = [
        c for c in binary_cols
        if c in section_map and section_map[c][0] == sec_name
    ]
    if sec_bin:
        pivot[sec_name + ' Score (%)'] = pivot[sec_bin].apply(
            lambda row: round(row[row != ''].sum() / len(sec_bin) * 100, 1),
            axis=1
        )

# ============================================================
# STEP 6 — SUB-SEGMENT TAGGING
# ============================================================
# Keywords used to classify each company into a sub-segment.
# Edit these lists to adjust the classification.

SEGMENTS = {
    'Luxury':         ['Hermes', 'Louis Vuitton', 'Prada', 'Burberry',
                       'Moncler', 'Brunello', 'TOD', 'Salvatore Ferragamo',
                       'Hugo Boss', 'Ralph Lauren'],
    'Fast Fashion':   ['H&M', 'Asos', 'Boohoo', 'Zara', 'Inditex',
                       'Primark', 'Shein', 'Esprit', 'Superdry',
                       'Urban Outfitters'],
    'Sports/Active':  ['Nike', 'Adidas', 'Puma', 'lululemon', 'Lululemon',
                       'Under Armour', 'Columbia', 'Asics', 'Skechers',
                       'JD Sports', 'Dick', 'Foot Locker', 'Hanesbrands',
                       'Gildan'],
    'Mid-Market':     ['Gap', 'Marks and Spencer', 'Next', 'Levi',
                       'American Eagle', 'Abercrombie', 'Express', 'Chico',
                       'Carter', 'Ted Baker', 'Gerry Weber', 'Guess',
                       'Zalando'],
    'Value/Dept':     ['Walmart', 'Target', 'Costco', 'Macy', 'Nordstrom',
                       'Burlington', 'Ross', 'Dillard', 'El Corte'],
    'Other Retail':   ['Amazon', 'Aldi', 'Carrefour', 'Otto',
                       'Sports Direct', 'Fossil', 'Buckle', 'DSW',
                       'Anta', 'Canada Goose', 'Semir', 'Capri',
                       'Childrens Place'],
}

def get_segment(company_name):
    for seg, keywords in SEGMENTS.items():
        if any(k.lower() in company_name.lower() for k in keywords):
            return seg
    return 'Other'

pivot.insert(1, 'Sub-segment', pivot['Company'].apply(get_segment))

# Sort by overall score descending
pivot = pivot.sort_values(
    'Overall Disclosure Score (%)', ascending=False
).reset_index(drop=True)

print(f"\nProcessing complete.")
print(f"  Final shape:    {pivot.shape}")
print(f"  Score range:    {pivot['Overall Disclosure Score (%)'].min()}% – "
      f"{pivot['Overall Disclosure Score (%)'].max()}%")

# ============================================================
# STEP 7 — BUILD EXCEL WORKBOOK
# ============================================================

print("\nBuilding Excel workbook...")
wb = Workbook()
all_cols = pivot.columns.tolist()
ncols    = len(all_cols)

## SHEET 1: FULL DATASET


In [ ]:
ws = wb.active
ws.title = 'Full Dataset'
ws.sheet_view.showGridLines = False
ws.freeze_panes = 'C4'   # freeze company + sub-segment columns

# Row 1 — Title bar
ws.merge_cells(f'A1:{get_column_letter(ncols)}1')
ws['A1'] = (
    'FASHION INDUSTRY SUSTAINABILITY DATASET  |  '
    'What Fuels Fashion — FTI 2024  |  '
    f'{len(pivot)} Companies  |  79 Metrics  |  '
    'Source: WikiRate / Fashion Revolution (CC BY 4.0)'
)
ws['A1'].font      = font(11, True, 'FFFFFF')
ws['A1'].fill      = fill(COLOURS['navy'])
ws['A1'].alignment = align('center')
ws.row_dimensions[1].height = 24

# Row 2 — Section header spans
ws['A2'].value = 'COMPANY';    _apply_section_header(ws, 'A2', 'A2', COLOURS['navy'])
ws['B2'].value = 'SUB-SEGMENT'; _apply_section_header(ws, 'B2', 'B2', COLOURS['navy'])

# Build section spans for columns C onwards
prev_sec   = None
sec_start  = None
prev_color = None

def _apply_section_header(ws, start_cell, end_cell, hex_color):
    """Not defined yet — using inline logic below instead."""
    pass

for i, col in enumerate(all_cols[2:], start=3):
    if col.endswith('Score (%)'):
        sec_name, sec_color = 'Summary Scores', COLOURS['teal']
    else:
        sec_name, sec_color = section_map.get(col, ('Other', COLOURS['slate']))

    if sec_name != prev_sec:
        # Write the previous section header
        if prev_sec and sec_start:
            end_i = i - 1
            if sec_start != end_i:
                ws.merge_cells(
                    start_row=2, start_column=sec_start,
                    end_row=2,   end_column=end_i
                )
            c = ws.cell(row=2, column=sec_start)
            c.value     = prev_sec
            c.font      = font(8, True, 'FFFFFF')
            c.fill      = fill(prev_color)
            c.alignment = align('center')
            c.border    = header_border()
        prev_sec   = sec_name
        prev_color = sec_color
        sec_start  = i

# Write the final section header
if sec_start:
    end_i = len(all_cols)
    if sec_start != end_i:
        ws.merge_cells(
            start_row=2, start_column=sec_start,
            end_row=2,   end_column=end_i
        )
    c = ws.cell(row=2, column=sec_start)
    c.value     = prev_sec
    c.font      = font(8, True, 'FFFFFF')
    c.fill      = fill(prev_color)
    c.alignment = align('center')
    c.border    = header_border()

ws.row_dimensions[2].height = 18

# Row 3 — Column headers
for j, col in enumerate(all_cols, 1):
    c = ws.cell(row=3, column=j)
    c.value = col
    if col in ('Company', 'Sub-segment'):
        col_color = COLOURS['navy']
    elif col.endswith('Score (%)'):
        col_color = COLOURS['teal']
    else:
        _, col_color = section_map.get(col, ('', COLOURS['slate']))
    c.font      = font(8, True, 'FFFFFF')
    c.fill      = fill(col_color)
    c.alignment = align('center', wrap=True)
    c.border    = header_border()
ws.row_dimensions[3].height = 75

# Data rows (start at row 4)
score_cols = [c for c in all_cols if c.endswith('Score (%)')]

for r_idx, (_, row) in enumerate(pivot.iterrows(), start=4):
    row_fill = fill(COLOURS['lt_blue']) if r_idx % 2 == 0 else fill(COLOURS['white'])

    for j, col in enumerate(all_cols, 1):
        val = row[col]
        c   = ws.cell(row=r_idx, column=j)
        c.value  = val
        c.border = thin_border()
        c.alignment = align('center')

        if col in ('Company', 'Sub-segment'):
            c.font = font(10, bold=(col == 'Company'))
            c.fill = row_fill

        elif col in score_cols:
            # Heatmap colouring for scores
            if isinstance(val, (int, float)):
                if val >= 60:
                    c.fill = fill(COLOURS['lt_green'])
                    c.font = font(10, True, COLOURS['green'])
                elif val >= 35:
                    c.fill = fill(COLOURS['yellow'])
                    c.font = font(10, True, 'E65100')
                else:
                    c.fill = fill(COLOURS['red_lt'])
                    c.font = font(10, True, 'B71C1C')
            else:
                c.fill = row_fill
                c.font = font(10)

        elif val == 1:
            # Yes = green
            c.fill = fill(COLOURS['lt_green'])
            c.font = font(9, True, COLOURS['green'])
        elif val == 0:
            # No = red
            c.fill = fill(COLOURS['red_lt'])
            c.font = font(9, True, 'B71C1C')
        else:
            c.fill = row_fill
            c.font = font(9)

    ws.row_dimensions[r_idx].height = 18

# Column widths
ws.column_dimensions['A'].width = 32
ws.column_dimensions['B'].width = 16
for j in range(3, ncols + 1):
    col = all_cols[j - 1]
    ws.column_dimensions[get_column_letter(j)].width = (
        16 if col.endswith('Score (%)') else 11
    )

## SHEET 2: CLUSTER INPUT


In [ ]:
ws2 = wb.create_sheet('Cluster Input (Summary)')
ws2.sheet_view.showGridLines = False
ws2.freeze_panes = 'C2'

# Select only the summary score columns for clustering
sum_cols = (
    ['Company', 'Sub-segment', 'Overall Disclosure Score (%)']
    + [s + ' Score (%)' for s in SECTIONS.keys()]
)
sum_df = pivot[sum_cols].copy()

# Title
ws2.merge_cells(f'A1:{get_column_letter(len(sum_cols))}1')
ws2['A1'] = (
    'CLUSTER ANALYSIS INPUT — Theme Scores per Company  |  '
    'Use these columns as features for K-means clustering'
)
ws2['A1'].font      = font(11, True, 'FFFFFF')
ws2['A1'].fill      = fill(COLOURS['navy'])
ws2['A1'].alignment = align('center')
ws2.row_dimensions[1].height = 24

# Header row
theme_colors = [
    COLOURS['teal'], COLOURS['navy'], COLOURS['navy'],
    '1B5E20', '0D47A1', 'BF360C', '880E4F', '37474F', '4A148C'
]
for j, col in enumerate(sum_cols, 1):
    c = ws2.cell(row=2, column=j)
    c.value     = col
    c.font      = font(9, True, 'FFFFFF')
    c.fill      = fill(theme_colors[min(j - 1, len(theme_colors) - 1)])
    c.alignment = align('center', wrap=True)
    c.border    = header_border()
ws2.row_dimensions[2].height = 50

# Data rows
for r_idx, (_, row) in enumerate(sum_df.iterrows(), start=3):
    row_fill = fill(COLOURS['lt_blue']) if r_idx % 2 == 0 else fill(COLOURS['white'])
    for j, col in enumerate(sum_cols, 1):
        val = row[col]
        c   = ws2.cell(row=r_idx, column=j)
        c.value  = val
        c.border = thin_border()
        c.alignment = align('center')

        if j <= 2:
            c.font = font(10, bold=(j == 1))
            c.fill = row_fill
        else:
            if isinstance(val, (int, float)):
                if val >= 60:
                    c.fill = fill(COLOURS['lt_green'])
                    c.font = font(10, True, COLOURS['green'])
                elif val >= 35:
                    c.fill = fill(COLOURS['yellow'])
                    c.font = font(10, True, 'E65100')
                else:
                    c.fill = fill(COLOURS['red_lt'])
                    c.font = font(10, True, 'B71C1C')
            else:
                c.fill = row_fill
                c.font = font(10)
    ws2.row_dimensions[r_idx].height = 20

ws2.column_dimensions['A'].width = 32
ws2.column_dimensions['B'].width = 16
for j in range(3, len(sum_cols) + 1):
    ws2.column_dimensions[get_column_letter(j)].width = 22

## SHEET 3: SEGMENT SUMMARY


In [ ]:
ws3 = wb.create_sheet('By Sub-segment')
ws3.sheet_view.showGridLines = False

seg_summary = (
    sum_df.groupby('Sub-segment')['Overall Disclosure Score (%)']
    .agg(['mean', 'min', 'max', 'count'])
    .round(1)
    .reset_index()
)
seg_summary.columns = [
    'Sub-segment', 'Avg Score (%)', 'Min Score (%)',
    'Max Score (%)', '# Companies'
]
seg_summary = seg_summary.sort_values('Avg Score (%)', ascending=False)

ws3.merge_cells('A1:E1')
ws3['A1'] = 'SUSTAINABILITY DISCLOSURE BY SUB-SEGMENT  |  Avg / Min / Max Overall Scores'
ws3['A1'].font      = font(12, True, 'FFFFFF')
ws3['A1'].fill      = fill(COLOURS['navy'])
ws3['A1'].alignment = align('center')
ws3.row_dimensions[1].height = 26

seg_hdr_colors = [COLOURS['navy'], COLOURS['teal'], '1B5E20', 'BF360C', '37474F']
for j, h in enumerate(seg_summary.columns, 1):
    c = ws3.cell(row=2, column=j)
    c.value     = h
    c.font      = font(10, True, 'FFFFFF')
    c.fill      = fill(seg_hdr_colors[j - 1])
    c.alignment = align('center', wrap=True)
    c.border    = header_border()
ws3.row_dimensions[2].height = 28

for r_idx, (_, row) in enumerate(seg_summary.iterrows(), start=3):
    for j, col in enumerate(seg_summary.columns, 1):
        val = row[col]
        c   = ws3.cell(row=r_idx, column=j)
        c.value  = val
        c.border = thin_border()
        c.alignment = align('center')
        if j == 1:
            c.font = font(10, True)
            c.fill = fill(COLOURS['lt_blue'])
        elif j == 2:
            c.font = font(11, True)
            if isinstance(val, (int, float)):
                if val >= 50:
                    c.fill = fill(COLOURS['lt_green'])
                    c.font = font(11, True, COLOURS['green'])
                elif val >= 30:
                    c.fill = fill(COLOURS['yellow'])
                    c.font = font(11, True, 'E65100')
                else:
                    c.fill = fill(COLOURS['red_lt'])
                    c.font = font(11, True, 'B71C1C')
        else:
            c.font = font(10)
            c.fill = fill(COLOURS['grey'])
    ws3.row_dimensions[r_idx].height = 24

for col_l, w in zip('ABCDE', [28, 16, 16, 16, 14]):
    ws3.column_dimensions[col_l].width = w

## TAB COLOURS


In [ ]:
ws.sheet_properties.tabColor  = '1A3C5E'
ws2.sheet_properties.tabColor = '00695C'
ws3.sheet_properties.tabColor = '880E4F'

## SAVE


In [ ]:
wb.save(OUTPUT_FILE)
print(f"\n✅ Saved: {OUTPUT_FILE}")
print("\n--- TOP 10 COMPANIES BY DISCLOSURE SCORE ---")
print(pivot[['Company', 'Sub-segment', 'Overall Disclosure Score (%)']].head(10).to_string(index=False))
print("\n--- SEGMENT SUMMARY ---")
print(seg_summary.to_string(index=False))